# Task 1 ? QLoRA fine-tuning for text-to-SQL

This notebook fine-tunes **Qwen2.5-0.5B-Instruct** on the SQL Create Context dataset. It uses a fixed 5,000/500/100 train/validation/test split, compares base and tuned exact match on the same held-out 100 examples, prints three tuned-model failures, and shows a 7B memory estimate. Run top to bottom on a Colab/Kaggle GPU; the first cell installs the libraries if needed.

In [1]:
%pip -q install -U "transformers>=4.45" "datasets>=2.20" "accelerate>=0.34" "peft>=0.13" "trl>=0.12" bitsandbytes

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [2]:
import gc, os, re, random
import numpy as np
import pandas as pd
import torch
import transformers, datasets, peft, trl
from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Enable a GPU runtime (Runtime > Change runtime type)."
print("GPU:", torch.cuda.get_device_name(0))
print("Versions:", transformers.__version__, datasets.__version__, peft.__version__, trl.__version__)

GPU: Tesla T4
Versions: 5.23.0 5.2.0 0.21.3 1.18.1


In [3]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # 0.5B parameters (within the <=1.5B limit)
DATASET_NAME = "b-mc2/sql-create-context"
OUTPUT_DIR = "/content/qwen-sql-qlora"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "adapter")
MAX_SEQUENCE_LENGTH = 1024
TRAIN_N, VALID_N, TEST_N = 5000, 500, 100
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LEARNING_RATE, NUM_EPOCHS = 2e-4, 3
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
SYSTEM_PROMPT = "You are an expert SQL query generator. Given a database schema and a natural language question, generate the correct SQL query. Return only the SQL query, without explanation or markdown."

In [4]:
raw = load_dataset(DATASET_NAME, split="train")
df = raw.to_pandas().dropna(subset=["context", "question", "answer"])
df = df.drop_duplicates(subset=["context", "question", "answer"]).reset_index(drop=True)
needed = TRAIN_N + VALID_N + TEST_N
if len(df) < needed:
    raise ValueError(f"Need at least {needed} unique rows for the required split; found {len(df)}")
# Hold out test first. Split remaining into train/validation, with stable seed.
all_ds = Dataset.from_pandas(df, preserve_index=False)
test_split = all_ds.train_test_split(test_size=TEST_N, seed=SEED, shuffle=True)
trainval = test_split["train"].train_test_split(test_size=VALID_N, seed=SEED, shuffle=True)
train_ds = trainval["train"].shuffle(seed=SEED).select(range(TRAIN_N))
valid_ds = trainval["test"]
test_ds = test_split["test"]
splits = DatasetDict(train=train_ds, validation=valid_ds, test=test_ds)
print({k:len(v) for k,v in splits.items()})

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

{'train': 5000, 'validation': 500, 'test': 100}


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def format_example(ex):
    messages = [
        {"role":"system", "content":SYSTEM_PROMPT},
        {"role":"user", "content":f"Database Schema:\n{ex['context']}\n\nQuestion:\n{ex['question']}\n\nGenerate the SQL query."},
        {"role":"assistant", "content":str(ex["answer"])},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

formatted = DatasetDict({k:v.map(format_example, remove_columns=v.column_names) for k,v in splits.items()})
print(formatted["train"][0]["text"][:1000])

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

<|im_start|>system
You are an expert SQL query generator. Given a database schema and a natural language question, generate the correct SQL query. Return only the SQL query, without explanation or markdown.<|im_end|>
<|im_start|>user
Database Schema:
CREATE TABLE table_name_61 (report VARCHAR, winning_driver VARCHAR)

Question:
Which Report has a Winning driver of emerson fittipaldi?

Generate the SQL query.<|im_end|>
<|im_start|>assistant
SELECT report FROM table_name_61 WHERE winning_driver = "emerson fittipaldi"<|im_end|>



## QLoRA setup and training

Rank 16 with alpha 32 provides a modest adapter capacity; targeting the attention projections adapts how schema and question tokens are related while keeping trainable weights small. A 2e-4 learning rate is a common starting point for LoRA adapters, and three epochs give a small model several passes over 5,000 examples without making training unnecessarily long. The 4-bit NF4 base weights plus 8-bit paged AdamW reduce GPU memory use; fp16 is selected for broad T4 compatibility.

In [6]:
# T4 (compute capability 7.5) cannot run TRL's BF16 chunked-logprob matmul path.
# Use BF16 only on Ampere (8.0) or newer; use FP16 on T4 and older GPUs.
gpu_major, gpu_minor = torch.cuda.get_device_capability(0)
compute_dtype = torch.bfloat16 if (gpu_major, gpu_minor) >= (8, 0) else torch.float16
print(f"GPU capability: {gpu_major}.{gpu_minor}; compute dtype: {compute_dtype}")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
    torch_dtype=compute_dtype,  # explicitly use FP16 on T4, even if model config prefers BF16
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES, bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
# Keep trainable adapter parameters in FP32. This avoids BF16 gradients reaching
# GradScaler on T4/older GPUs while the frozen 4-bit base computes in FP16.
for parameter in model.parameters():
    if parameter.requires_grad and parameter.dtype != torch.float32:
        parameter.data = parameter.data.float()
base_dtypes = sorted({str(p.dtype) for name,p in model.named_parameters() if "lora_" not in name})
adapter_dtypes = sorted({str(p.dtype) for p in model.parameters() if p.requires_grad})
print("Backbone parameter dtypes:", base_dtypes)
print("Trainable adapter dtypes:", adapter_dtypes)
if gpu_major < 8 and any("bfloat16" in dtype for dtype in base_dtypes):
    raise RuntimeError("T4 detected but BF16 backbone weights remain. Restart runtime and run from the top.")
if any(p.dtype != torch.float32 for p in model.parameters() if p.requires_grad):
    raise RuntimeError("Expected FP32 trainable adapter parameters before training.")
model.print_trainable_parameters()

GPU capability: 7.5; compute dtype: torch.float16


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3195: FutureWarning: `gradient_checkpointing_enable` is deprecated and will be removed in v6. Use `activation_checkpointing_enable` instead.
  warnings.warn(


Backbone parameter dtypes: ['torch.float32', 'torch.uint8']
Trainable adapter dtypes: ['torch.float32']
trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


In [7]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=0,  # compatible with older SFTConfig releases
    lr_scheduler_type="cosine",
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    # Do not enable AMP/GradScaler here: on Colab T4 the scaler can receive
    # BF16 gradients from the quantized/model path and raise NotImplementedError.
    # The frozen 4-bit backbone still computes in compute_dtype (FP16 on T4).
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_length=MAX_SEQUENCE_LENGTH,
    packing=False,
    dataset_text_field="text",
    loss_type="nll",  # avoid TRL chunked_logprob BF16 kernel on pre-Ampere GPUs
    report_to="none",
    seed=SEED,
)
trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=formatted["train"], eval_dataset=formatted["validation"],
    processing_class=tokenizer,
)
# SFTTrainer/PEFT may recast adapters while initializing. Set final trainable
# parameter dtype after construction, before Trainer creates its optimizer.
for parameter in trainer.model.parameters():
    if parameter.requires_grad and parameter.dtype != torch.float32:
        parameter.data = parameter.data.float()
print("Trainer precision:", {"fp16": trainer.args.fp16, "bf16": trainer.args.bf16})
print("Trainable dtypes:", sorted({str(p.dtype) for p in trainer.model.parameters() if p.requires_grad}))
assert trainer.args.fp16 is False and trainer.args.bf16 is False, "Restart runtime and rerun config with AMP disabled."
assert all(p.dtype == torch.float32 for p in trainer.model.parameters() if p.requires_grad), "A trainable parameter is not FP32 after trainer initialization."
result = trainer.train()
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(result.metrics)
print("Saved adapter:", ADAPTER_DIR)

/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

... (4 outputs left out)

Dropping fully masked examples from eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Trainer precision: {'fp16': False, 'bf16': False}
Trainable dtypes: ['torch.float32']


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.588067,0.568468,0.580009,0.869677,604738.000000
2,0.533731,0.551695,0.551168,0.872827,1209476.000000
3,0.525650,0.549115,0.535997,0.873218,1814214.000000


{'train_runtime': 2139.7906, 'train_samples_per_second': 7.01, 'train_steps_per_second': 0.439, 'total_flos': 4528280167065600.0, 'train_loss': 0.5758456168210291, 'epoch': 3.0}
Saved adapter: /content/qwen-sql-qlora/adapter


## Held-out evaluation: base vs fine-tuned

Metric: **SQL normalized exact match**. Each prediction and reference is lowercased, stripped, and whitespace-normalized, with one trailing semicolon ignored. This is intentionally strict: semantically equivalent SQL with different structure can count as incorrect. Both models see the same 100 untouched test rows and use greedy decoding.

In [8]:
def make_prompt(ex):
    messages = [
        {"role":"system", "content":SYSTEM_PROMPT},
        {"role":"user", "content":f"Database Schema:\n{ex['context']}\n\nQuestion:\n{ex['question']}\n\nGenerate the SQL query."},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def normalize_sql(text):
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text[:-1].rstrip() if text.endswith(";") else text

def predict(model, ex):
    prompt = make_prompt(ex)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                                pad_token_id=tokenizer.eos_token_id)
    generated = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def evaluate(model, data):
    preds, refs = [], []
    model.eval()
    for ex in data:
        preds.append(predict(model, ex))
        refs.append(str(ex["answer"]))
    scores = [normalize_sql(p) == normalize_sql(r) for p,r in zip(preds,refs)]
    return {"exact_match": float(np.mean(scores)), "correct": int(sum(scores)),
            "total":len(scores), "predictions":preds, "references":refs}

# Evaluate the base model before attaching the adapter.
del trainer, model
gc.collect(); torch.cuda.empty_cache()
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
    torch_dtype=compute_dtype,  # explicitly use FP16 on T4, even if model config prefers BF16
)
base_result = evaluate(base_model, splits["test"])
print(f"Base: {base_result['correct']}/{base_result['total']} = {base_result['exact_match']:.1%}")
del base_model
gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Base: 5/100 = 5.0%


In [9]:
from peft import PeftModel
base_for_adapter = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
    torch_dtype=compute_dtype,  # explicitly use FP16 on T4, even if model config prefers BF16
)
tuned_model = PeftModel.from_pretrained(base_for_adapter, ADAPTER_DIR)
tuned_result = evaluate(tuned_model, splits["test"])
print(f"Fine-tuned: {tuned_result['correct']}/{tuned_result['total']} = {tuned_result['exact_match']:.1%}")
print(f"Change: {(tuned_result['exact_match']-base_result['exact_match'])*100:+.1f} percentage points")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


Fine-tuned: 66/100 = 66.0%
Change: +61.0 percentage points


In [10]:
# Show up to three real held-out failures (if fewer fail, show all available failures).
failures = []
for i, (ex, pred, ref) in enumerate(zip(splits["test"], tuned_result["predictions"], tuned_result["references"])):
    if normalize_sql(pred) != normalize_sql(ref):
        failures.append({"test_index":i, "question":ex["question"], "schema":ex["context"],
                         "prediction":pred, "reference":ref})
    if len(failures) == 3:
        break
if not failures:
    print("No exact-match failures on this run; inspect the next three test examples as challenging cases.")
    failures = [{"test_index":i, "question":splits["test"][i]["question"],
                 "schema":splits["test"][i]["context"], "prediction":tuned_result["predictions"][i],
                 "reference":tuned_result["references"][i]} for i in range(min(3,len(splits["test"])))]
for n, f in enumerate(failures, 1):
    print(f"\nFailure {n} (test row {f['test_index']})")
    print("Question:", f["question"])
    print("Prediction:", f["prediction"])
    print("Reference:", f["reference"])
    print("Likely cause: inspect the schema and SQL difference. Typical causes are ambiguous wording, confusing similar columns/tables, or a join/filter/aggregation detail; the strict exact-match metric also penalizes structurally different but equivalent SQL.")


Failure 1 (test row 2)
Question: Who did the Texan's play on Week 4?
Prediction: SELECT opponent FROM table_name_37 WHERE week = "week 4"
Reference: SELECT opponent FROM table_name_37 WHERE week = "4"
Likely cause: inspect the schema and SQL difference. Typical causes are ambiguous wording, confusing similar columns/tables, or a join/filter/aggregation detail; the strict exact-match metric also penalizes structurally different but equivalent SQL.

Failure 2 (test row 3)
Question: Which Points have Touchdowns larger than 0, and an Extra points smaller than 0?
Prediction: SELECT AVG(points) FROM table_name_70 WHERE touchdowns > 0 AND extra_points < 0
Reference: SELECT SUM(points) FROM table_name_70 WHERE touchdowns > 0 AND extra_points < 0
Likely cause: inspect the schema and SQL difference. Typical causes are ambiguous wording, confusing similar columns/tables, or a join/filter/aggregation detail; the strict exact-match metric also penalizes structurally different but equivalent SQL.



## Approximate GPU memory for fine-tuning a 7B model

Working estimate, decimal GB (1 GB = 10? bytes), excluding activations and temporary buffers first; actual peak memory depends on sequence length, batch size, optimizer, checkpointing, and implementation. For full Adam fine-tuning, assume 16-bit weights plus 16-bit gradients, and FP32 master weights plus two FP32 Adam states: **2 + 2 + 4 + 8 = 16 bytes/parameter**. For LoRA, the frozen base is 16-bit (2 bytes/parameter); adapter optimizer state is small relative to the base. For QLoRA, the base is 4-bit (0.5 bytes/parameter), with quantization metadata adding overhead.

In [11]:
params = 7_000_000_000
full_weights = params * 16 / 1e9
lora_base = params * 2 / 1e9
qlora_base = params * 0.5 / 1e9
print(f"Full fine-tuning model states: 7B ? 16 bytes = {full_weights:.0f} GB (~{full_weights/1.073741824:.1f} GiB), plus activations/workspace")
print(f"LoRA frozen base only: 7B ? 2 bytes = {lora_base:.0f} GB (~{lora_base/1.073741824:.1f} GiB), plus small adapter states and activations")
print(f"QLoRA quantized base only: 7B ? 0.5 bytes = {qlora_base:.1f} GB (~{qlora_base/1.073741824:.1f} GiB), plus quantization metadata, adapters, activations/workspace")
print("Typical planning ranges: full Adam fine-tuning roughly 100?120+ GB; LoRA on a 16-bit base often roughly 20?30+ GB; QLoRA often roughly 8?16+ GB. These are rough peak-memory ranges, not guarantees.")

Full fine-tuning model states: 7B ? 16 bytes = 112 GB (~104.3 GiB), plus activations/workspace
LoRA frozen base only: 7B ? 2 bytes = 14 GB (~13.0 GiB), plus small adapter states and activations
QLoRA quantized base only: 7B ? 0.5 bytes = 3.5 GB (~3.3 GiB), plus quantization metadata, adapters, activations/workspace
Typical planning ranges: full Adam fine-tuning roughly 100?120+ GB; LoRA on a 16-bit base often roughly 20?30+ GB; QLoRA often roughly 8?16+ GB. These are rough peak-memory ranges, not guarantees.
